In [ ]:

import pandas as pd
import numpy as np

def evaluate_test_control(df):
    """
    Avalia a similaridade entre os grupos test e control.

    Colunas esperadas:
    dt_ref, store_id, net_sales, gross_sales, qty_sold, flag
    """

    df = df.copy()

    # Garantir formato de data
    df['dt_ref'] = pd.to_datetime(df['dt_ref'])

    # Agrupar métricas por dia e grupo
    df_daily = (
        df.groupby(['dt_ref', 'flag'], as_index=False)
        .agg(
            net_sales=('net_sales', 'sum'),
            gross_sales=('gross_sales', 'sum'),
            qty_sold=('qty_sold', 'sum'),
            n_stores=('store_id', 'nunique')
        )
    )

    # Criar séries comparativas
    metrics = [
        'net_sales',
        'gross_sales',
        'qty_sold'
    ]

    comparison = {}

    for metric in metrics:

        pivot = (
            df_daily
            .pivot(
                index='dt_ref',
                columns='flag',
                values=metric
            )
            .reindex(columns=['test', 'control'])
            .fillna(0)
        )

        pivot.columns = [
            f'{metric}_test',
            f'{metric}_control'
        ]

        pivot[f'{metric}_difference'] = (
            pivot[f'{metric}_control'] -
            pivot[f'{metric}_test']
        )

        pivot[f'{metric}_abs_difference'] = (
            pivot[f'{metric}_difference'].abs()
        )

        pivot[f'{metric}_ape'] = (
            pivot[f'{metric}_abs_difference'] /
            pivot[f'{metric}_test'].abs().replace(0, np.nan)
        ) * 100

        comparison[metric] = pivot

    # Consolidar as métricas em um único dataframe
    df_comparison = pd.concat(
        comparison.values(),
        axis=1
    ).reset_index()

    # Calcular indicadores gerais
    results = []

    for metric in metrics:

        test = df_comparison[f'{metric}_test']
        control = df_comparison[f'{metric}_control']
        abs_difference = df_comparison[f'{metric}_abs_difference']

        denominator = test.abs().sum()

        results.append({
            'metric': metric,
            'MAE': abs_difference.mean(),
            'MAPE': df_comparison[f'{metric}_ape'].mean(),
            'WAPE': (
                abs_difference.sum() / denominator * 100
                if denominator != 0 else np.nan
            ),
            'Correlation': (
                test.corr(control)
                if test.nunique() > 1 and control.nunique() > 1
                else np.nan
            )
        })

    df_metrics = pd.DataFrame(results)

    return df_comparison, df_metrics


In [ ]:
df_comparison, df_metrics = evaluate_test_control(df_result)

In [ ]:

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. IDENTIFICAR AS LOJAS DE CADA GRUPO
control_stores = df_state['store_id'].unique().tolist()
test_stores = df_state['store_id'].unique().tolist()

# 2. FILTRAR O DATAFRAME ORIGINAL
df_plot = df_state.loc[df_state['store_id'].isin(list(control_stores) + list(test_stores))].copy()
df_plot['group'] = df_plot['store_id'].map({**{store: 'Control' for store in control_stores}, **{store: 'Test' for store in test_stores}})

# 3. AGRUPAR AS VENDAS POR DIA E GRUPO
df_daily = (df_plot.groupby(['dt_ref', 'group'], as_index=False)['net_sales'].sum())
df_daily = (df_daily.pivot(index='dt_ref',columns='group',values='net_sales').fillna(0).reset_index())
df_daily = df_daily.sort_values('dt_ref')

# 4. CALCULAR AS DIFERENÇAS
df_daily['difference'] = (df_daily['Control'] - df_daily['Test'])
df_daily['difference_pct'] = (df_daily['difference'] / df_daily['Test'].replace(0, float('nan'))) * 100

# 5. CRIAR O GRÁFICO
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12, subplot_titles=(
        'Vendas diárias: Grupo Teste vs. Controle',
        'Diferença percentual entre os grupos'
    ), row_heights=[0.65, 0.35]
)

# Vendas do grupo teste & controle
fig.add_trace(go.Scatter(x=df_daily['dt_ref'], y=df_daily['Test'], name='Grupo Teste', mode='lines+markers', line=dict(width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_daily['dt_ref'], y=df_daily['Control'], name='Grupo Controle', mode='lines+markers', line=dict(width=2)), row=1, col=1)

# Diferença percentual
fig.add_trace(go.Bar(x=df_daily['dt_ref'], y=df_daily['difference_pct'], name='Diferença percentual', hovertemplate='%{y:.2f}%<extra></extra>'), row=2, col=1)

# Linha de referência em zero
fig.add_hline(y=0, line_dash='dash', line_width=1, line_color='gray', row=2, col=1)

fig.update_layout(
    title='Comparação dos grupos experimentais',
    template='plotly_white',
    hovermode='x unified',
    height=750,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    )
)

fig.update_yaxes(title_text='Vendas (R$)', tickformat=',.2f', row=1, col=1)
fig.update_yaxes(title_text='Diferença (%)', ticksuffix='%', row=2, col=1)
fig.update_xaxes(title_text='Data', row=2, col=1)
fig.show()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[
        'Net Sales',
        'Gross Sales',
        'Quantity Sold'
    ],
    vertical_spacing=0.10
)

metrics_config = [
    ('net_sales', 'Net Sales'),
    ('gross_sales', 'Gross Sales'),
    ('qty_sold', 'Quantity Sold')
]

for i, (metric, label) in enumerate(metrics_config, start=1):

    fig.add_trace(
        go.Scatter(
            x=df_comparison['dt_ref'],
            y=df_comparison[f'{metric}_test'],
            name='Test',
            legendgroup='test',
            showlegend=(i == 1),
            mode='lines',
        ),
        row=i, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df_comparison['dt_ref'],
            y=df_comparison[f'{metric}_control'],
            name='Control',
            legendgroup='control',
            showlegend=(i == 1),
            mode='lines',
        ),
        row=i, col=1
    )

fig.update_layout(
    title='Test vs Control — Daily Performance',
    template='plotly_white',
    height=850,
    hovermode='x unified'
)

fig.show()